---
title: "Basketball Position Classification & Inference"
author: "Andratx Bellmunt"
abstract: >
    We use NBA official player stats data to build classification models to predict their playing positions. A supervised model lets us derive a rule to assign hybrid positions to pure positions and to infer non-listed values. An unsupervised model is used to evaluate potential new positions based on playing style rather than using traditional labels.
format:
  html:
    code-fold: true
    self-contained: true
    html-math-method: katex
    include-after-body: _tracker.html
jupyter: python3
number-sections: false
toc: true
toc-depth: 2
toc-title: Contents
---

# Foreword

The purpose of this report is to create a proof of concept on how machine learning classification models can be used with sports data, in particular to **study player positions in team sports**.

We choose to test the idea using **basketball data** for a combination of reasons:

- Teams are small and there is a limited number of positions, which makes data suitable for a small study

- The National Basketball Association (NBA) provides easy programmatic access to their official data

- Personal interest of the author in basketball as a former practitioner and coach

We aim at answering two main questions:

- Are we able to **predict positions** from player stats?

- Based on player stats, can we **build new positions** than reflect better what happens in-game rather than assigning traditional labels?

To address the first question we will use a supervised learning model (Random Forest Classifier) and to answer the second question we use an unsupervised learning model (KMeans).

Finally, note that detailed model selection and fine-tuning is out of the scope for this report. We just run out-of-the-box versions of models that are known to be suitable for the task at hand. Our focus will be on the analysis that illustrates the concept.

# Initialization

In this section we import all the required libraries and we configure some notebook-wise parameters.

In [ ]:
# General imports
import time
from typing import Tuple

# Data Processing
import pandas as pd

# NBA API
from nba_api.stats.endpoints import leaguedashplayerstats, commonteamroster
from nba_api.stats.static import teams

# Statistics
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report

# Plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

In [ ]:
# Render plots compatible with HTML
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Display 2 decimal positions in dataframes
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# Get raw data

We pull the data directly from the NBA official API:

- We use data from **10 regular seasons**: from 2016-17 to 2025-26 both included.

- We use the `leaguedashplayerstats` endpoint to pull player stats and the `commonteamroster` endpoint to get player positions (the latter is more efficient than making and individual call for each player).

> *N.B.:* The NBA API is known to be strict with calls. We strongly recommend storing a hard copy of the data once it is pulled from the API. That was our option of choice and our code reflects that choice. However, code on how to call the API is still included for reference.

In [ ]:
def get_player_stats(first_season: int, last_season: int) -> pd.DataFrame:
    assert first_season <= last_season, 'Last season must be later than first season'

    # Initialize a list to append data for each season
    list_seasons_data = []

    for season in range(first_season, last_season + 1):
        print(f"Fetching {season}...")

        # Initialize list to append data for each type of measure
        list_measure_data = []

        # Call API
        stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season=season,
            season_type_all_star='Regular Season',
            measure_type_detailed_defense='Base'
        )

        # Store the data
        df_stats = stats.get_data_frames()[0]

        # Drop _RANK columns
        # since they just represent position by different criteria
        for col in df_stats.columns:
            if "RANK" in col:
                df_stats.drop(col, axis=1, inplace=True)

        # Sleep to not overload API calls
        time.sleep(0.5)

        # Add SEASON column to identify data when we concatenate all seasons together
        df_stats['SEASON'] = season

        # Append data to the list of dataframes for all seasons
        list_seasons_data.append(df_stats)

    # Concatenate the data together and set an ID for each player + season combination   
    df = pd.concat(list_seasons_data, ignore_index=True)
    df['ID'] = df['PLAYER_ID'].astype(str) + '_' + df['SEASON'].astype(str)

    return df

In [ ]:
def get_player_positions(first_season: int, last_season: int) -> pd.DataFrame:
    all_teams = teams.get_teams()
    rosters = []

    for team in all_teams:
        for season in range(first_season, last_season + 1):
            roster = commonteamroster.CommonTeamRoster(
                team_id=team['id'],
                season=season
            )
            df_roster = roster.get_data_frames()[0]
            df_roster['SEASON'] = season

            rosters.append(roster.get_data_frames()[0])
            time.sleep(0.5)  # avoid rate limiting

    df = pd.concat(rosters)[['PLAYER_ID', 'SEASON', 'POSITION']]
    df['ID'] = df['PLAYER_ID'].astype(str) + '_' + df['SEASON'].astype(str)

    return df[['ID', 'POSITION']]

In [ ]:
df_stats_raw = pd.read_csv("../assets/nba_player_stats.csv")
# df_stats_raw = get_player_stats(first_season=2016, last_season=2025)

In [ ]:
df_positions_raw = pd.read_csv("../assets/nba_player_positions.csv")
# df_positions_raw = get_player_positions(first_season=2016, last_season=2025)

In [ ]:
df_raw = df_stats_raw.merge(df_positions_raw, how='left')

# Data cleaning

Before getting into modeling we need to clean our data. We want to focus on players with relevant playing time and consider only core playing statistics as predictors of position. To that end the cleaning function applies the following rules:

- **Records with relevant sample size:** we look only at players with at least 250 minutes of play (equivalent to at least 10 games with 25 minutes of play each).

- **In-game statistics only:** we drop all columns that are not direct metrics of player performance during games.

- **Avoid correlations:** Drop columns that are obviously correlated with core playing statistics.

- **Normalization:** we normalized all metrics (except for rates) to avoid volume bias. We follow the standard of giving stats per 36 minutes.

- **Parse percentage columns:** To help readability whe remove the "_PCT" from shooting percentages columns.

- **Parse non-listed position:** Some players do not have a listed position in our data set. We label them simply as "not_listed".

In [ ]:
def clean_raw_data(df: pd.DataFrame) -> pd.DataFrame:
    # Filter out players with low playing time
    df = df[df['MIN'] >= 250]

    # Drop all columns that are not in-game statistics
    not_in_game_cols = [
        'PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'ID', 'AGE',
        'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_COUNT', 
        'SEASON', 'GP', 'W', 'L', 'W_PCT'
    ]
    df = df.drop(columns=not_in_game_cols)

    # Drop metrics that are correlated with core metrics
    correlated_cols = [
        'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA',  # Redundant with shooting percentages
        'OREB', 'DREB',                              # Accounted for through REB
        'PLUS_MINUS',                                # Highly dependent on team quality
        'DD2', 'TD3',                                # Double-doubles and triple-doubles direct result of other metrics
        'NBA_FANTASY_PTS', 'WNBA_FANTASY_PTS'        # Computed directly from a formula using other stats
    ]
    df = df.drop(columns=correlated_cols)

    # Normalize (per 36 minutes)
    for col in ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'BLKA', 'PF', 'PFD']:
        df[col] = df[col] / df['MIN'] * 36

    # Drop unneeded column after normalizing
    df = df.drop(columns='MIN')

    # Parse percentage columns
    for col in ['FT', 'FG', 'FG3']:
        df = df.rename(columns={f'{col}_PCT': col})

    # Parse unlisted positions
    df['POSITION'] = df['POSITION'].fillna('not_listed')

    return df

In [ ]:
df = clean_raw_data(df_raw)

Now that our data is clean we shall make a minimal inspection to understand what are the predicting stats and how does the target look like:

In [ ]:
print("Player stats:")
print([col for col in df.columns if col != 'POSITION'])
print()
print("List of positions:")
print(sorted(df['POSITION'].unique()))

Note that we obtained a **collection of 12 different statistics** to predict positions: they include shooting percentages, rebounds, assists, turnovers, steals, blocks, blocks against, personal fouls, personal fouls driven and points.

It is interesting to see that aside from the **pure positions** Guard (G), Forward (F) and Center (C) there exist **hybrid positions** that combine two of the pure positions. Also there are some players with no listed position. Both hybrid and not-listed positions will become very interesting in model evaluation.

In [ ]:
# Define global variables for features and positions given in a convenient ordering
FEATURES = [
    'FG', 'FG3', 'FT',
    'PTS', 'REB', 'AST', 
    'STL', 'BLK', 'PFD',
    'TOV', 'BLKA', 'PF', 
]
POSITIONS=['G', 'F', 'C', 'G-F', 'F-G', 'F-C', 'C-F', 'not_listed']

# Supervised learning

The maing goal of this section is to build a classification model that is able to **predict the position of a player from his in-game stats.**

In order to keep things clean we will **train only for pure positions (G, F, C)**. We set aside hybrid positions and not-listed positions, which we study later once the model is built.

In [ ]:
df_pure = df[df['POSITION'].isin(['G', 'F', 'C'])].copy()
df_hybrid = df[~df['POSITION'].isin(['G', 'F', 'C'])].copy()  # Includes not-listed position as well

## Data preparation

Split pure position data into training and test datasets.

In [ ]:
def pure_position_train_test_split(
    df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # Isolate features (X) from target (y)
    X = df.drop(columns='POSITION')
    y = df['POSITION']

    # Split train and test
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=13,  # Ensures reproducible results
        stratify=y        # Stratify to avoid class imbalance
    )

    # Build training and testing dataframes
    df_train = X_train.copy()
    df_train['POSITION'] = y_train

    df_test = X_test.copy()
    df_test['POSITION'] = y_test

    return df_train, df_test

In [ ]:
df_pure_train, df_pure_test = pure_position_train_test_split(df_pure)

## Exploratory Data Analysis

As it is common practice, we will study our training data to understand how predictors (player stats) relate to the target (position) and also possible correlations between them to judge if we need to clean the data further.

In [ ]:
def plot_features_boxplots(df: pd.DataFrame, target: str='POSITION') -> None:
    # Initialize a 3x4 subplot grid
    fig = make_subplots(
        rows=4,
        cols=3,
        subplot_titles=FEATURES,  # Dynamic titles matching feature names
        vertical_spacing=0.08,    # Space out rows to keep text readable
        horizontal_spacing=0.06
    )

    # Iterate through features and map them to row/col indices
    for i, feature in enumerate(FEATURES):
        # Plotly grid indices are 1-based (not 0-based)
        row = (i // 3) + 1
        col = (i % 3) + 1

        # Add boxplot trace to specific grid coordinate
        fig.add_trace(
            go.Box(
                x=df[target],
                y=df[feature],
                name="",                    # Hides redundant individual names on x-axis
                boxpoints="outliers",       # Options: 'all', 'outliers', or False
                notched=False,
                marker_color="#1f77b4"    # Custom thematic styling choice
            ),
            row=row,
            col=col
        )

    # Optimize layout configuration for complex grid densities
    fig.update_layout(
        height=1200,
        width=800,
        title_text="Statistical Profiling: 12-Feature Boxplot Matrix",
        showlegend=False,           # Traces are self-evident
        template="plotly_dark"
    )

    # Render interactive dashboard
    fig.show()

plot_features_boxplots(df_pure_train)

The boxplot shows the statistical distribution of each feature with respect to each position. We can see that, in general, everything aligns to common basketball knowledge (analysis is given per row):

- **Shooting percentages:** Centers play closer to the basket and therefore their field-goal percentage (`FG`) is clearly higher. This is reversed when it comes to 3-point field-goal percentage (`FG3`) because guards and forwards are much better at long-range shooting. Also free-throw percentage (`FT`) gradually lowers from guards to centers as smaller player tend to be more accurate shooters.

- **Positive stats (I):** Points (`PTS`) are similar accross positions, but rebounds (`REB`) and assists (`AST`) clearly correlate with distance to the basket: frontcourt players grab more rebounds while more playmakers are found in the backcourt.

- **Positive stats (II):** We see a similar patter were backcourt players steal (`STL`) more balls while frontcourt players are in a position to block (`BLK`) more shots. Personal fouls driven (`PFD`) has a lower correlation with target.

- **Negative stats:** When it comes to negative stats centers show a more clear defensive effort and make more fouls (`PF`). The number of lost balls (`TOV`) and blocks against (`BLKA`) have a lesser predictive power since all players suffer them at more similar levels.

Overall we have several metrics that seem great predictors of the positions. However, we have to be careful with collinearity because some features might be giving redundant information if they are very correlated.

In [ ]:
print("Correlation Matrix for Pure Position Players (Train Set)")
display(
    df_pure_train[FEATURES]
    .corr()
    .style
    .background_gradient(cmap='coolwarm')
    .set_properties(width='50px')
    .format('{:.2f}')
    #.set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}])
    .set_table_styles([
        # Force column width and right-alignment on data cells (td)
        {'selector': 'td', 'props': [('width', '50px'), ('text-align', 'right'), ('padding', '5px')]},
        # Force column width and center-alignment on headers (th)
        {'selector': 'th', 'props': [('width', '50px'), ('text-align', 'center'), ('padding', '5px')]},
        # Optional: Prevent cell content from wrapping awkwardly
        {'selector': 'table', 'props': [('table-layout', 'fixed'), ('width', 'auto')]}
    ])

)

As it was already hinted there are some metrics that are essentially telling us the same story. E.g. in the group given by `FG`, `REB`, `BLK` there are high correlations and correspond to the idea "this player plays close to the basket and he is a center".

In order to asses the severity of these correlations we compute Variance Inflation Factors (VIF), which instead of looking at pairwise correlations it regresses each feature against all the others to quantify how inflanted its variance is.

In [ ]:
def compute_vif(df: pd.DataFrame) -> pd.DataFrame:
    # Take the features
    dfn = df[FEATURES].copy()

    # Statsmodels VIF requires an explicit intercept (constant) to avoid forcing
    # the regression through the origin (0,0), which severely inflates VIF math.
    dfn = add_constant(dfn)

    # Calculate VIF for each feature
    df_vif = pd.DataFrame()
    df_vif["feature"] = dfn.columns
    df_vif["VIF"] = [
        variance_inflation_factor(dfn.values, i) for i in range(dfn.shape[1])
    ]
    df_vif = df_vif.query('feature != "const"')

    return df_vif.sort_values(by="VIF", ascending=False)

compute_vif(df_pure_train).style.hide(axis=0).format({'VIF': '{:.2f}'})

Since all the VIFs are below 5, we will not drop any feature for our training. None of them is deemed redundant enough to be ruled out.

## Model training

In [ ]:
# For consistency with the later unsupervised model we create a pipeline
supervised_pipeline = make_pipeline(RandomForestClassifier(random_state=8))

# Train the supervised model
supervised_pipeline.fit(df_pure_train[FEATURES], df_pure_train['POSITION'])

## Model evaluation

In [ ]:
print("Confusion Matrix for Pure Position Players")
display(
    pd.crosstab(
        df_pure_test['POSITION'],
        supervised_pipeline.predict(df_pure_test[FEATURES]),
        rownames=['Actual'],
        colnames=['Predicted']
    )[['G', 'F', 'C']].reindex(['G', 'F', 'C'])
)

In [ ]:
print("Classification report for supservised learning model")
print(
    classification_report(
        df_pure_test['POSITION'],
        supervised_pipeline.predict(df_pure_test[FEATURES]),
        target_names=['G', 'F', 'C']
    )
)

In [ ]:
print("Feature importance")
display(
    pd.DataFrame({
        'Feature': FEATURES,
        'Importance (%)': 100 * supervised_pipeline.named_steps['randomforestclassifier'].feature_importances_
    }).sort_values(by='Importance (%)', ascending=False).style.hide(axis=0).format({'Importance (%)': '{:.2f}'})
)

With an 86% accuracy rate and a balanced 0.85 macro F1-score, the model does a fantastic job of cutting through the noise of modern basketball styles. It proves most accurate when identifying Centers (90% precision), which makes complete sense: true big-man metrics like blocks and rebounds remain highly distinct. The slight drop in Forward precision (82%) reflects real-world basketball, capturing those 'tweener' wings who naturally blur the lines betIen guards and frontcourt players.

Key takeaways:

- **Center Dominance:** The model nails Centers. A 0.90 precision means when the model says a player is a Center, it is right 90% of the time. Frontcourt players still leave a very distinct statistical footprint. Note that 3 of the top 4 importance features (`REB`, `BLK`, `FG`) are precisely the ones that we signaled during exploratory data analysis as highly-correlated indicators of the Center position.

- **Guard Vacuum:** Guards have a high recall (0.91), meaning the model catches almost every single guard in the dataset. However, its lower precision (0.80) tells us that some players who are actually Forwards are putting up guard-like numbers and getting swept into the Guard category. Assists (`AST`) is the main feature to identify Guards.

## Post-Supervised Inference & Hybrid Position Analysis

Now that the supervised model is deployed we can use to predict core positions (G, F, C) for the two subsets the we put aside: players listed with hybrid designations (e.g., G-F) and players with completely missing positional labels:

In [ ]:
print("Prediction Matrix for Hybrid Position (and not-listed) Players")
display(
    pd.crosstab(
        df_hybrid['POSITION'],
        supervised_pipeline.predict(df_hybrid[FEATURES]),
        rownames=['Actual'],
        colnames=['Predicted'],
    )[['G', 'F', 'C']].reindex(['G-F', 'F-G', 'F-C', 'C-F', 'not_listed'])
)

Several key insights emerge from these results:

- **Boundary Precision:** The model demonstrated exceptional semantic consistency when evaluating hybrid players. Only 3 players listed as G-F/F-G Ire classified as Centers, and conversely, only 3 players listed as F-C/C-F were classified as Guards. This incredibly low rate of extreme errors confirms the model maintains structural integrity, rarely confusing the opposite ends of the positional spectrum.

- **Deciphering the Hybrid Nomenclature:** A compelling pattern emerged regarding primary designations: a distinct majority of G-F players were classified as Guards, while F-G players predominantly mapped to Forwards. This trend mirrored perfectly across the frontcourt, where F-C players leaned toward Forwards and C-F players mapped to Centers. This strongly validates the hypothesis that **the order of letters in mixed positions is highly deliberate**, where the first letter denotes the player's primary or preferred role. Consequently, using the first initial of a hybrid tag serves as a statistically sound proxy for isolating pure positions.

- **Positional Inference for Unlabeled Rows:** For players entirely lacking a listed position, the model predicted a distribution of **53% Guards, 35% Forwards, and 12% Centers**. This aligns remarkably well with the baseline distribution observed across the training and test datasets (50% / 38% / 12%). Given this structural consistency and the model's proven F1-score (0.85), these predicted labels can be confidently adopted as ground-truth proxies to eliminate missing values in the master dataset.

# Unsupervised learning

## Model training

To build the unsupervised model we take KMeans with 3 clusters (to mimic the fact that we have 3 pure positions). Since this model is based on distances, we rescale the data before training:

In [ ]:
# Create a copy of the data to avoid overwriting
dfu = df.copy()

# Build an unsupervised pipeline
unsupervised_pipeline = make_pipeline(
    StandardScaler(),  # Fits and transforms across the entire player landscape
    KMeans(n_clusters=3, random_state=8, n_init=10)
)

# Generate playstyle clusters
dfu['CLUSTER'] = unsupervised_pipeline.fit_predict(dfu[FEATURES])

## Model evaluation

In order to make a thorough evaluation of the unsupervised model we look into two different aspects of it:

- How clusters relate with the existing labels: we provide a prediction matrix and a bar chart visual.

- How the predicting metrics distribute across the clusters: we provide detailed statistical distribution tables and also a more visual analysis in the form of boxplots. We also look into centroids per metric to understand what stats were more relevant to the model when building the clusters.

In [ ]:
print("Prediction Matrix for Mixed Position Players")
display(pd.crosstab(dfu['POSITION'], dfu['CLUSTER']).reindex(POSITIONS))

In [ ]:
def plot_clusters_against_target(df: pd.DataFrame) -> None:
    # Copy data
    dfc = df.copy()

    # Build and normalize the cross-tabulation matrix (Percentages per row)
    crosstab = pd.crosstab(dfc['POSITION'], dfc['CLUSTER'])
    crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100

    # Reorder the rows to match our basketball layout sequence
    crosstab_pct = crosstab_pct.reindex(POSITIONS)

    # Initialize figure
    fig = go.Figure()

    # Define a distinct color palette for the 3 clusters
    cluster_colors = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c'}

    # 4. Add each cluster as an independent stacked trace
    for cluster_id in [0, 1, 2]:
        # Pull percentages across positions for this specific cluster ID
        y_percentages = crosstab_pct[cluster_id].values

        fig.add_trace(
            go.Bar(
                x=POSITIONS,
                y=y_percentages,
                name=f"Cluster {cluster_id}",
                marker_color=cluster_colors.get(cluster_id, '#7f7f7f'),
                hovertemplate="<b>Position:</b> %{x}<br>" +
                              "<b>Proportion:</b> %{y:.1f}%<br>" +
                              "<extra></extra>"
            )
        )

    # 5. Enforce stacking and styling properties
    fig.update_layout(
        title={
            'text': "Playstyle Cluster Distributions Across Listed Positions",
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title="Listed Position Profile",
        yaxis_title="Proportion of Position Category (%)",
        barmode='stack',
        template='plotly_dark',
        yaxis_ticksuffix='%',
        yaxis_range=[0, 100],
        legend_title_text="Assigned Cluster",
        height=500,
        width=800
    )

    fig.show()

plot_clusters_against_target(dfu)

In [ ]:
for cluster in [0, 1, 2]:
    print(f'Features statistical distributions for Cluster {cluster}')
    display(dfu[dfu['CLUSTER']==cluster][FEATURES].describe())
    print()

In [ ]:
plot_features_boxplots(dfu, target='CLUSTER')

In [ ]:
print("Cluster Centroids Z-scores (standard deviations relative to the league average)")
display(
    pd.DataFrame(
        unsupervised_pipeline.named_steps['kmeans'].cluster_centers_,  # Centroids represent Z-scores because we used scaling
        columns=FEATURES
    ).T
)

#### Cluster 0: The Primary Initiators & High-Usage Superstars
- **Positional Composition:** Contains 615 pure Guards, 193 Forwards, and 27 Centers, alongside a smattering of hybrid designations.

- **Statistical Signature:** This cluster represents the elite offensive engines of the league. It is characterized by an exceptional scoring load (Mean: 20.85 `PTS`; Centroid: +1.07) and dominant playmaking responsibility (Mean: 5.70 `AST`; Centroid: +1.10). This massive offensive volume naturally correlates with higher ball security risks (Mean: 2.82 `TOV`; Centroid: +1.16) and an aggressive capacity to draw defensive friction (Mean: 4.03 `PFD`; Centroid: +0.93). Defensively, they remain light on interior metrics (Mean: 0.51 `BLK`; Centroid: -0.37) but are highly efficient perimeter shooters, sitting well above the league mean in free-throw efficiency (Mean: 80% `FT`; Centroid: +0.42).

- **Archetype Profile:** This cluster captures the league's high-usage superstars and primary ball-handlers. The model isolated these players not strictly by their baseline defensive position, but by their sheer volume of offensive creation, playmaking execution, and defensive attention drawn.

#### Cluster 1: The Low-Usage Baseline & System Spacers
-  **Positional Composition:** The largest cluster in the dataset (2,323 players), capturing the absolute majority of pure Guards (939), Forwards (802), and perimeter-dominant hybrids (289 G-F, 116 F-G).

- **Statistical Signature:** This massive group represents the literal baseline of the modern league. Strikingly, their centroid reveals **negative Z-scores across almost every major box score category**, including scoring (Mean: 13.55 `PTS`; Centroid: -0.42), playmaking (Mean: 2.97 `AST`; Centroid: -0.26), and turnovers (Mean: 1.54 `TOV`; Centroid: -0.51). However, they maintain positive value in three-point spacing (Mean: 35% `FG3`; Centroid: +0.28), while showing below-average efficiency everywhere else inside the arc (Mean: 43% `FG`; Centroid: -0.43).

-  **Archetype Profile:** This cluster represents the modern system player and depth rotation asset. These are the supporting guards, wings, and "3-and-D" specialists who provide vital spacing from beyond the arc without demanding heavy isolation usage, high minutes, or primary playmaking duties.

#### Cluster 2: The Paint Anchors & Interior Specialists
- **Positional Composition:** Tightly constrained to the frontcourt, holding 335 pure Centers and the vast majority of frontcourt hybrids (154 F-C, 140 C-F), while tracking virtually zero perimeter guards (only 10).

-  **Statistical Signature:** A flawless mathematical representation of absolute interior dominance. This cluster isolates the league's elite rebounders (Mean: 10.79 `REB`; Centroid: +1.40) and primary rim protectors (Mean: 1.58 `BLK`; Centroid: +1.32). Their offensive profile consists entirely of high-efficiency looks near the rim, yielding a massive surge in overall efficiency (Mean: 56% `FG`; Centroid: +1.30). Conversely, they completely abandon the perimeter, sitting a full standard deviation below the league average in three-point shooting (Mean: 21% `FG3`; Centroid: -1.01) and free throws (Mean: 68% `FT`; Centroid: -0.82). Their physical role results in intense defensive friction (Mean: 4.16 `PF`; Centroid: +1.06).

-  **Archetype Profile:** This is the traditional frontcourt baseline. The unsupervised algorithm naturally recognized that despite the fluid nature of positionless basketball, the physical signatures of rim protection, box-outs, and interior efficiency remain completely distinct from the rest of the league.

# Methodological limitation

Before concluding, it is vital to acknowledge a fundamental causal limitation inherent in using performance data to classify or cluster basketball positions. While both the supervised and unsupervised models demonstrate high mathematical accuracy in aligning box score statistics to positional labels, we must confront a critical question: **Are we measuring natural archetypes, or are we simply mapping pre-determined tactical constraints?**

In data science, this is known as a **behavioral feedback loop**. When a player enters the league labeled as a "Center," coaching staffs immediately prescribe specific operational boundaries: they are assigned to guard the opposing big man, positioned near the rim on offense, expected to prioritize screen-setting and rebounding, and historically discouraged from taking perimeter shots. 

Consequently, the player’s statistical footprint—high field goal percentages, high rebound volumes, and minimal three-point attempts—is heavily pre-determined by their **prescribed role**, rather than just their organic playstyle.

Therefore, our models are not entirely discovering independent, objective truths about player capability. Instead, they are measuring the structural rigidity of basketball coaching and tactical design. A macro F1-score of 0.85 does not just prove that "Guards play differently than Centers"; it proves that coaches still strictly enforce traditional positional responsibilities on the court, generating the very data used to validate those positions in a circular loop.

This feedback loop is exactly why **Cluster 0 (The Elite Offensive Engine)** is so fascinating. By pulling a small mix of Centers (27) and Forwards (193) into a predominantly Guard-driven cluster, the unsupervised model successfully identified instances where modern talent broke free from traditional coaching constraints, prioritizing raw individual creation volume over pre-labeled positional tracking.